# DataCo Supply Chain Analytics
### End-to-End Data Analytics Portfolio Project

**Author:** Vrushang Gheewala  
**Dataset:** [DataCo Smart Supply Chain — Kaggle](https://www.kaggle.com/datasets/shashwatwork/dataco-smart-supply-chain-for-big-data-analysis)  
**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Tableau Public

---

## Project Overview
This notebook covers the full data pipeline for analysing 180,000+ supply chain orders across global markets.

**Pipeline Steps:**
1. Load & explore raw data
2. Clean columns and parse dates
3. Calculate delay days and SLA breach flag
4. Flag order statuses
5. Feature engineering
6. Null handling
7. Segment groupby aggregation
8. EDA visualisations
9. Export clean CSVs for Tableau

**Output Files:**
- `supply_chain_master.csv` — used for Dashboards 1 & 2
- `segment_profit.csv` — used for Dashboard 3

## Step 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', 55)
pd.set_option('display.float_format', '{:.2f}'.format)

print('Libraries loaded successfully ✅')

## Step 2 — Load & Explore Raw Data

In [ ]:
# Load dataset — latin1 encoding handles Spanish special characters
df = pd.read_csv('DataCoSupplyChainDataset.csv', encoding='latin1')

print(f'Shape: {df.shape}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')
df.head()

In [ ]:
# Check data types and null counts
df.info()

In [ ]:
# Check null values per column
nulls = df.isnull().sum()
print('Columns with nulls:')
print(nulls[nulls > 0])

## Step 3 — Clean Columns

In [ ]:
# Strip whitespace from column names
df.columns = df.columns.str.strip()

# Drop PII and irrelevant columns
cols_to_drop = [
    'Customer Email', 'Customer Password', 'Customer Street',
    'Customer Zipcode', 'Customer Fname', 'Customer Lname',
    'Product Description', 'Product Image'
]
df.drop(columns=[c for c in cols_to_drop if c in df.columns], inplace=True)

print(f'Columns after drop: {df.shape[1]}')
print('PII columns removed ✅')

## Step 4 — Parse Dates

In [ ]:
# Parse order date
df['order date (DateOrders)'] = pd.to_datetime(df['order date (DateOrders)'])

# Extract date components
df['order_month']   = df['order date (DateOrders)'].dt.month
df['order_year']    = df['order date (DateOrders)'].dt.year
df['order_quarter'] = df['order date (DateOrders)'].dt.quarter

print('Date parsing complete ✅')
print(f'Year range: {df["order_year"].min()} — {df["order_year"].max()}')
print(f'Unique months: {df["order_month"].nunique()}')

## Step 5 — Calculate Delay Days & SLA Breach

In [ ]:
# Delay = actual shipping days minus scheduled shipping days
# Positive = late, Zero = on time, Negative = early
df['delay_days'] = df['Days for shipping (real)'] - df['Days for shipment (scheduled)']

# SLA breach flag
df['sla_breach'] = df['delay_days'] > 0

print('Delay calculation complete ✅')
print(f'On time orders: {(~df["sla_breach"]).sum():,}')
print(f'Late orders: {df["sla_breach"].sum():,}')
print(f'SLA breach rate: {df["sla_breach"].mean():.1%}')

## Step 6 — Flag Order Statuses

In [ ]:
# Clean status column
df['order_status_clean'] = df['Order Status'].str.strip().str.upper()

# Create status flags — keep all orders, just flag each type
df['is_cancelled']  = df['order_status_clean'] == 'CANCELED'
df['is_completed']  = df['order_status_clean'].isin(['COMPLETE', 'CLOSED'])
df['is_active']     = df['order_status_clean'].isin([
    'PENDING', 'PROCESSING', 'ON_HOLD',
    'PAYMENT_REVIEW', 'PENDING_PAYMENT'
])
df['is_fraud']      = df['order_status_clean'] == 'SUSPECTED_FRAUD'

print('Order status flags created ✅')
print(df['order_status_clean'].value_counts())

## Step 7 — Feature Engineering

In [ ]:
# Total order value = price × quantity
df['total_order_value'] = df['Order Item Product Price'] * df['Order Item Quantity']

# Profit margin percentage
df['profit_margin_pct'] = (df['Order Profit Per Order'] / df['Sales']) * 100

print('Feature engineering complete ✅')
print(f'Avg profit margin: {df["profit_margin_pct"].mean():.2f}%')
print(f'Avg order value: ${df["total_order_value"].mean():,.2f}')

## Step 8 — Handle Nulls

In [ ]:
# Check nulls in key columns before export
key_cols = [
    'Sales', 'Order Profit Per Order',
    'Days for shipping (real)', 'Days for shipment (scheduled)',
    'delay_days', 'sla_breach'
]

print('Nulls in key columns:')
print(df[key_cols].isnull().sum())

# Drop rows with nulls in key columns
rows_before = df.shape[0]
df.dropna(subset=key_cols, inplace=True)
rows_after = df.shape[0]

print(f'\nRows dropped: {rows_before - rows_after:,}')
print(f'Final row count: {rows_after:,}')

## Step 9 — EDA Visualisations

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('DataCo Supply Chain — EDA Overview', fontsize=14, fontweight='bold')

# Chart 1 — Monthly Revenue Trend
monthly = df.groupby('order_month')['Sales'].sum()
axes[0].fill_between(monthly.index, monthly.values, alpha=0.3, color='#4A7C59')
axes[0].plot(monthly.index, monthly.values, color='#4A7C59', linewidth=2)
axes[0].set_title('Monthly Revenue Trend')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Revenue (USD)')
axes[0].grid(False)

# Chart 2 — Delay Days Distribution
delay_counts = df['delay_days'].value_counts().sort_index()
colours = ['#4A7C59' if x <= 0 else '#C8852A' if x <= 3 else '#E8705A' 
           for x in delay_counts.index]
axes[1].bar(delay_counts.index, delay_counts.values, color=colours)
axes[1].set_title('Delay Days Distribution')
axes[1].set_xlabel('Delay Days')
axes[1].set_ylabel('Number of Orders')
axes[1].grid(False)

# Chart 3 — Top 10 Categories by Revenue
top_cats = df.groupby('Category Name')['Sales'].sum().nlargest(10)
axes[2].barh(top_cats.index[::-1], top_cats.values[::-1], color='#4A7C59')
axes[2].set_title('Top 10 Categories by Revenue')
axes[2].set_xlabel('Revenue (USD)')
axes[2].grid(False)

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA charts saved ✅')

## Step 10 — Segment Aggregation

In [ ]:
# Group by Customer Segment for Dashboard 3
df_seg = df.groupby('Customer Segment').agg(
    total_revenue       = ('Sales', 'sum'),
    total_profit        = ('Order Profit Per Order', 'sum'),
    avg_order_value     = ('Sales', 'mean'),
    order_count         = ('Order Id', 'count'),
    late_delivery_pct   = ('sla_breach', 'mean')
).reset_index()

# Convert late delivery to percentage
df_seg['late_delivery_pct'] = df_seg['late_delivery_pct'] * 100

print('Segment aggregation complete ✅')
df_seg

## Step 11 — Export Clean CSVs for Tableau

In [ ]:
# Export master CSV — used for Dashboards 1 & 2
df.to_csv('supply_chain_master.csv', index=False)
print(f'supply_chain_master.csv exported ✅ — {df.shape[0]:,} rows, {df.shape[1]} columns')

# Export segment CSV — used for Dashboard 3
df_seg.to_csv('segment_profit.csv', index=False)
print(f'segment_profit.csv exported ✅ — {df_seg.shape[0]} rows, {df_seg.shape[1]} columns')

## Summary

| Output | Description | Used In |
|---|---|---|
| `supply_chain_master.csv` | Full cleaned dataset with engineered features | Dashboards 1 & 2 |
| `segment_profit.csv` | Aggregated by Customer Segment | Dashboard 3 |
| `eda_overview.png` | EDA summary charts | README |

### Key Stats
- **Total Orders:** 180,519
- **SLA Breach Rate:** ~57%
- **Avg Profit Margin:** ~10.8%
- **Top Category:** Fishing ($6.9M revenue)
- **Worst Delivery Mode:** First Class (95% late rate)

---
*For dashboard visualisations, see Tableau Public links in README.md*